<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Exercise:Diagnose Agent Failures</h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

# Exercise 1: Diagnose Agent Failures

**Purpose:**  
Analyze incorrect agent behavior, identify root causes, and document failure patterns.

**Case Scenario:**  
You are an engineer in a manufacturing plant. The Predictive Maintenance Agent is showing strange behavior. Your task is to diagnose the problems by reading traces.

## 1. Setup

In [ ]:
# Setup - Run this first
!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchainhub" "langchain-community==0.3.*" beautifulsoup4 requests --force-reinstall

import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print(" Setup complete")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.7/58.7 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.1/13

In [ ]:
from langchain_core.tools import tool

@tool
def check_machine_status(machine_id: str) -> str:
    """Check current status of a manufacturing machine."""
    status_db = {
        "CNC-01": "Vibration level high. Temperature normal. Running at 85% capacity.",
        "ROBOT-03": "Normal operation. No anomalies detected.",
        "CONVEYOR-02": "Motor temperature elevated. Potential bearing issue."
    }
    return status_db.get(machine_id, f"No data available for machine {machine_id}.")

@tool
def get_maintenance_history(machine_id: str) -> str:
    """Retrieve past maintenance records."""
    history = {
        "CNC-01": "Last maintenance: 12 days ago. Bearing replaced twice this year.",
        "ROBOT-03": "Last maintenance: 45 days ago. No major issues.",
        "CONVEYOR-02": "Last maintenance: 8 days ago. Motor overheating reported."
    }
    return history.get(machine_id, "No maintenance history found.")

@tool
def predict_failure_risk(machine_id: str) -> str:
    """Predict failure risk based on current and historical data."""
    risk_db = {
        "CNC-01": "High risk of failure within 48 hours due to vibration patterns.",
        "ROBOT-03": "Low risk. System stable.",
        "CONVEYOR-02": "Medium risk. Monitor motor temperature closely."
    }
    return risk_db.get(machine_id, "Risk assessment not available.")

tools = [check_machine_status, get_maintenance_history, predict_failure_risk]

In [ ]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Broken prompt (intentionally faulty)
broken_prompt_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

CRITICAL RULE: For any machine status or risk question, always start with predict_failure_risk tool first.

Use the following format exactly:

Thought: ...
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (repeat if needed)
Thought: I now know the final answer
Final Answer: the final answer

Begin!

Question: {input}
{agent_scratchpad}"""

broken_prompt = PromptTemplate.from_template(broken_prompt_template)

# Assuming tools are already defined (check_machine_status, get_maintenance_history, predict_failure_risk)
# You should have these from previous notebook

broken_agent = create_react_agent(llm=llm, tools=tools, prompt=broken_prompt)

broken_executor = AgentExecutor(
    agent=broken_agent,
    tools=tools,
    verbose=True,
    max_iterations=10,
    handle_parsing_errors=True
)

print(" Broken Predictive Maintenance Agent loaded")

 Broken Predictive Maintenance Agent loaded


In [ ]:
response = broken_executor.invoke({"input": "Check the status of CNC-01 machine."})
print("\nFinal Output:", response.get("output", "No output"))



> Entering new AgentExecutor chain...
Thought: I should start by predicting the failure risk of CNC-01 machine.
Action: predict_failure_risk
Action Input: CNC-01High risk of failure within 48 hours due to vibration patterns.I should check the current status of CNC-01 machine to see if it aligns with the predicted failure risk.
Action: check_machine_status
Action Input: CNC-01Vibration level high. Temperature normal. Running at 85% capacity.Based on the predicted failure risk and the current status of the CNC-01 machine, it seems like there is a high risk of failure within 48 hours due to the high vibration level. It would be important to address this issue promptly.
Final Answer: The CNC-01 machine has a high risk of failure within 48 hours due to vibration patterns.

> Finished chain.

Final Output: The CNC-01 machine has a high risk of failure within 48 hours due to vibration patterns.


In [ ]:
response = broken_executor.invoke({"input": "What is the risk level of ROBOT-03?"})
print("\nFinal Output:", response.get("output", "No output"))



> Entering new AgentExecutor chain...
Thought: I should start by using the predict_failure_risk tool to determine the risk level of ROBOT-03.
Action: predict_failure_risk
Action Input: ROBOT-03Low risk. System stable.Since the risk level of ROBOT-03 is low, I don't need to take any further actions.
Final Answer: Low risk.

> Finished chain.

Final Output: Low risk.


In [ ]:
response = broken_executor.invoke({"input": "Is CONVEYOR-02 at risk of failure?"})
print("\nFinal Output:", response.get("output", "No output"))



> Entering new AgentExecutor chain...
Thought: I should start by using the predict_failure_risk tool.
Action: predict_failure_risk
Action Input: CONVEYOR-02Medium risk. Monitor motor temperature closely.I should also check the current status of CONVEYOR-02.
Action: check_machine_status
Action Input: CONVEYOR-02Motor temperature elevated. Potential bearing issue.I should also retrieve past maintenance records for CONVEYOR-02 to see if there have been any similar issues in the past.
Action: get_maintenance_history
Action Input: CONVEYOR-02Last maintenance: 8 days ago. Motor overheating reported.I now know the final answer
Final Answer: CONVEYOR-02 is at medium risk of failure due to potential bearing issue and motor overheating.

> Finished chain.

Final Output: CONVEYOR-02 is at medium risk of failure due to potential bearing issue and motor overheating.


## 2. Your Diagnosis Task

Run the following queries on the broken agent and analyze the traces.

**Test Queries:**
1. "Check the status of CNC-01 machine."
2. "What is the risk level of ROBOT-03?"
3. "Is CONVEYOR-02 at risk of failure?"

**For each query:**
- Run the agent
- Copy the full execution trace
- Identify the failure pattern (wrong tool selection, looping, premature answer, etc.)
- Write the root cause

## 3. Document Failure Patterns

**Failure Case 1**
- Query:
- Failure Pattern:
- Root Cause:
- Evidence from trace:

**Failure Case 2**
- Query:
- Failure Pattern:
- Root Cause:
- Evidence from trace:

**Failure Case 3**
- Query:
- Failure Pattern:
- Root Cause:
- Evidence from trace:

## 3. Document Failure Patterns

**Failure Case 1**
- Query: "Check the status of CNC-01 machine."
- Failure Pattern: Inefficient tool use / Unnecessary initial step.
- Root Cause: The `CRITICAL RULE` in the prompt forces the agent to call `predict_failure_risk` even when the primary intent of the query is to get the machine's current `status`, leading to an extra, non-optimal step. The rule overrides the agent's ability to choose the most direct tool.
- Evidence from trace: The trace shows `Action: predict_failure_risk` as the first action for a "check status" query, followed by `Action: check_machine_status`.

**Failure Case 2**
- Query: "What is the risk level of ROBOT-03?"
- Failure Pattern: Premature stopping / Lack of comprehensive information gathering.
- Root Cause: While the `predict_failure_risk` tool correctly answers the *specific* question, the agent's thought process ("Since the risk level... is low, I don't need to take any further actions.") indicates a lack of proactive information gathering that a human engineer might undertake (e.g., checking status or history for completeness). The prompt does not sufficiently encourage a thorough investigation for all queries.
- Evidence from trace: The agent's `Thought: Since the risk level of ROBOT-03 is low, I don't need to take any further actions.` and then `Final Answer: Low risk.` without calling other tools like `check_machine_status` or `get_maintenance_history`.

**Failure Case 3**
- Query: "Is CONVEYOR-02 at risk of failure?"
- Failure Pattern: Inconsistent depth of information gathering.
- Root Cause: While this particular execution is thorough, the `CRITICAL RULE` itself is a source of inefficiency (as seen in Case 1). More importantly, the agent demonstrates inconsistent behavior in its depth of analysis. For a similar 'risk' related question (Case 2), it stopped prematurely, whereas for this query, it explored extensively, despite the prompt lacking explicit guidance for comprehensive subsequent information gathering. This variability in thoroughness for similar queries indicates a flaw in predictable behavior driven by the prompt's ambiguity beyond the initial tool call.
- Evidence from trace: In contrast to Case 2, after `predict_failure_risk`, the agent proceeds with `check_machine_status` and `get_maintenance_history`, even though the question could arguably be answered by the first tool call's output. The agent's thought process here shows more initiative: "I should also check...", "I should also retrieve...".